Before starting, you will need to install some packages to reproduce the baseline.

In [1]:
!pip install tqdm
!pip install scikit-learn

In [2]:
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

## Data loading

In [3]:
# put your own path to the data root directory (see example in `Data architecture` section)
data_dir = Path("/Users/enfants/Code/OWKIN_ML2")

# load the training and testing data sets
train_features_dir = data_dir / "train_input" / "moco_features"
test_features_dir = data_dir / "test_input" / "moco_features"
df_train = pd.read_csv(data_dir  / "supplementary_data" / "train_metadata.csv")
df_test = pd.read_csv(data_dir  / "supplementary_data" / "test_metadata.csv")

# concatenate y_train and df_train
y_train = pd.read_csv(data_dir  / "train_output.csv")
df_train = df_train.merge(y_train, on="Sample ID")

print(f"Training data dimensions: {df_train.shape}")  # (344, 4)
df_train.head()

Training data dimensions: (344, 4)


,Sample ID,Patient ID,Center ID,Target
0,ID_001.npy,P_001,C_1,0
1,ID_002.npy,P_002,C_2,1
2,ID_005.npy,P_005,C_5,0
3,ID_006.npy,P_006,C_5,0
4,ID_007.npy,P_007,C_2,1


## Data processing

In [4]:
X_train = []
y_train = []
centers_train = []
patients_train = []

for sample, label, center, patient in tqdm(
    df_train[["Sample ID", "Target", "Center ID", "Patient ID"]].values
):
    # load the coordinates and features (1000, 3+2048)
    _features = np.load(train_features_dir / sample)
    # get coordinates (zoom level, tile x-coord on the slide, tile y-coord on the slide)
    # and the MoCo V2 features
    coordinates, features = _features[:, :3], _features[:, 3:]  # Ks
    # slide-level averaging
    X_train.append(np.mean(features, axis=0))
    y_train.append(label)
    centers_train.append(center)
    patients_train.append(patient)

# convert to numpy arrays
X_train = np.array(X_train)
y_train = np.array(y_train)
centers_train = np.array(centers_train)
patients_train = np.array(patients_train)

100%|██████████| 344/344 [00:01<00:00, 327.63it/s]


In [5]:
# /!\ we perform splits at the patient level so that all samples from the same patient
# are found in the same split

patients_unique = np.unique(patients_train)
y_unique = np.array(
    [np.mean(y_train[patients_train == p]) for p in patients_unique]
)
centers_unique = np.array(
    [centers_train[patients_train == p][0] for p in patients_unique]
)

print(
    "Training set specifications\n"
    "---------------------------\n"
    f"{len(X_train)} unique samples\n"
    f"{len(patients_unique)} unique patients\n"
    f"{len(np.unique(centers_unique))} unique centers"
)

Training set specifications
---------------------------
344 unique samples
305 unique patients
3 unique centers


In [6]:
aucs = []
lrs = []
# 5-fold CV is repeated 5 times with different random states
for k in range(5):
    kfold = StratifiedKFold(5, shuffle=True, random_state=k)
    fold = 0
    # split is performed at the patient-level
    for train_idx_, val_idx_ in kfold.split(patients_unique, y_unique):
        # retrieve the indexes of the samples corresponding to the
        # patients in `train_idx_` and `test_idx_`
        train_idx = np.arange(len(X_train))[
            pd.Series(patients_train).isin(patients_unique[train_idx_])
        ]
        val_idx = np.arange(len(X_train))[
            pd.Series(patients_train).isin(patients_unique[val_idx_])
        ]
        # set the training and validation folds
        X_fold_train = X_train[train_idx]
        y_fold_train = y_train[train_idx]
        X_fold_val = X_train[val_idx]
        y_fold_val = y_train[val_idx]
        # instantiate the model
        lr = LogisticRegression(C=0.01, solver="liblinear")
        # fit it
        lr.fit(X_fold_train, y_fold_train)
        # get the predictions (1-d probability)
        preds_val = lr.predict_proba(X_fold_val)[:, 1]
        # compute the AUC score using scikit-learn
        auc = roc_auc_score(y_fold_val, preds_val)
        print(f"AUC on split {k} fold {fold}: {auc:.3f}")
        aucs.append(auc)
        # add the logistic regression to the list of classifiers
        lrs.append(lr)
        fold += 1
    print("----------------------------")
print(
    f"5-fold cross-validated AUC averaged over {k+1} repeats: "
    f"{np.mean(aucs):.3f} ({np.std(aucs):.3f})"
)

AUC on split 0 fold 0: 0.590
AUC on split 0 fold 1: 0.490
AUC on split 0 fold 2: 0.673
AUC on split 0 fold 3: 0.705
AUC on split 0 fold 4: 0.536
----------------------------
AUC on split 1 fold 0: 0.695
AUC on split 1 fold 1: 0.667
AUC on split 1 fold 2: 0.498
AUC on split 1 fold 3: 0.502
AUC on split 1 fold 4: 0.558
----------------------------
AUC on split 2 fold 0: 0.568
AUC on split 2 fold 1: 0.730
AUC on split 2 fold 2: 0.593
AUC on split 2 fold 3: 0.524
AUC on split 2 fold 4: 0.517
----------------------------
AUC on split 3 fold 0: 0.718
AUC on split 3 fold 1: 0.661
AUC on split 3 fold 2: 0.535
AUC on split 3 fold 3: 0.471
AUC on split 3 fold 4: 0.447
----------------------------
AUC on split 4 fold 0: 0.563
AUC on split 4 fold 1: 0.535
AUC on split 4 fold 2: 0.697
AUC on split 4 fold 3: 0.455
AUC on split 4 fold 4: 0.607
----------------------------
5-fold cross-validated AUC averaged over 5 repeats: 0.581 (0.087)


# Attention based MIL

In [7]:
# ==========================================
# 1) Data prep (MIL bags) & 3) Modèle Attention MIL
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Dataset PyTorch
class MILDataset(Dataset):
    def __init__(self, bags, labels, patient_ids):
        self.bags = bags
        self.labels = labels
        self.patient_ids = patient_ids
        
    def __len__(self):
        return len(self.patient_ids)
        
    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        bag = self.bags[pid]  # numpy array (n_tiles, d)
        label = self.labels[pid]
        return torch.tensor(bag, dtype=torch.float32), torch.tensor([label], dtype=torch.float32), pid

def collate_fn(batch):
    # batch is a list of tuples: (bag_tensor, label_tensor, pid)
    bags = [item[0] for item in batch]
    labels = torch.stack([item[1] for item in batch])
    pids = [item[2] for item in batch]
    return bags, labels, pids

# Modèle Attention MIL (Ilse et al.)
class AttentionMIL(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=256):
        super(AttentionMIL, self).__init__()
        
        # Tile encoder
        self.tile_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Attention mechanism (Gated attention for better performance)
        self.attention_V = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh()
        )
        self.attention_U = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Sigmoid()
        )
        self.attention_weights = nn.Linear(128, 1)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, x):
        # x is assumed to be of shape (n_tiles, input_dim) for a single bag
        
        # Encoder: H shape -> (n_tiles, hidden_dim)
        H = self.tile_encoder(x)
        
        # Attention
        A_V = self.attention_V(H)  # (n_tiles, 128)
        A_U = self.attention_U(H)  # (n_tiles, 128)
        A = self.attention_weights(A_V * A_U)  # (n_tiles, 1)
        A = torch.transpose(A, 1, 0)  # (1, n_tiles)
        A = F.softmax(A, dim=1)  # Softmax over tiles
        
        # Bag embedding: Z shape -> (1, hidden_dim)
        Z = torch.mm(A, H)
        
        # Classification
        logits = self.classifier(Z)  # (1, 1)
        
        return logits, A


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/miniconda3/envs/churn/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/miniconda3/envs/churn/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/miniconda3/envs/churn/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/opt/miniconda3/envs/churn/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.s

Using device: mps


In [9]:
# ==========================================
# 2) Normalisation, 4) Training loop & 5) CV groupée patient
# ==========================================
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import torch.optim as optim
from sklearn.metrics import roc_auc_score

print("Building train bags (without averaging tiles)...")

# Building bags: For each patient, we stack all tiles from all their samples
bags_train_all = defaultdict(list)
labels_train_all = {}

for sample, label, patient in tqdm(df_train[["Sample ID", "Target", "Patient ID"]].values, desc="Train Bags"):
    _features = np.load(train_features_dir / sample)
    features = _features[:, 3:]  # (1000, 2048)
    bags_train_all[patient].append(features)
    labels_train_all[patient] = label

# Concatenate tiles per patient
for p in bags_train_all:
    bags_train_all[p] = np.concatenate(bags_train_all[p], axis=0)

bags_test = defaultdict(list)
for sample, patient in tqdm(df_test[["Sample ID", "Patient ID"]].values, desc="Test Bags"):
    _features = np.load(test_features_dir / sample)
    features = _features[:, 3:]
    bags_test[patient].append(features)

for p in bags_test:
    bags_test[p] = np.concatenate(bags_test[p], axis=0)


# Function to train one epoch
def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    all_labels = []
    all_preds = []
    for bags, labels, pids in dataloader:
        optimizer.zero_grad()
        
        # We loop over bags since batch contains variable shapes
        batch_loss = 0
        for bag, label in zip(bags, labels):
            bag = bag.to(device)
            label = label.to(device)
            
            logits, A = model(bag)
            loss = criterion(logits, label.unsqueeze(0))
            batch_loss += loss
            
            all_labels.append(label.item())
            all_preds.append(torch.sigmoid(logits).item())
            
        batch_loss = batch_loss / len(bags)
        batch_loss.backward()
        optimizer.step()
        total_loss += batch_loss.item()
        
    auc = roc_auc_score(all_labels, all_preds) if len(np.unique(all_labels)) > 1 else 0.5
    return total_loss / len(dataloader), auc

# Function to evaluate
def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    all_labels = []
    all_preds = []
    all_pids = []
    
    with torch.no_grad():
        for bags, labels, pids in dataloader:
            batch_loss = 0
            for bag, label in zip(bags, labels):
                bag = bag.to(device)
                label = label.to(device)
                
                logits, A = model(bag)
                loss = criterion(logits, label.unsqueeze(0))
                batch_loss += loss
                
                all_labels.append(label.item())
                all_preds.append(torch.sigmoid(logits).item())
                all_pids.append(pids)
                
            total_loss += batch_loss.item() / len(bags)
            
    auc = roc_auc_score(all_labels, all_preds) if set(all_labels) == {0.0, 1.0} else 0.5
    return total_loss / len(dataloader), auc, all_preds, all_pids

print("\nStarting 5-fold CV Training...")
cv_aucs = []
oof_preds = {}
oof_labels = {}
patience = 5
max_epochs = 15
models = []  # save models for inference

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kfold.split(patients_unique, y_unique)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    train_patients = patients_unique[train_idx]
    val_patients = patients_unique[val_idx]
    
    # 2) Normalization (Fit on Train fold ONLY)
    scaler = StandardScaler()
    # Flatten train bags to fit scaler
    train_features_flat = np.concatenate([bags_train_all[p] for p in train_patients], axis=0)
    scaler.fit(train_features_flat)
    del train_features_flat
    
    # Transform bags
    fold_bags_train = {p: scaler.transform(bags_train_all[p]) for p in train_patients}
    fold_bags_val = {p: scaler.transform(bags_train_all[p]) for p in val_patients}
    
    # Datasets and loaders
    train_ds = MILDataset(fold_bags_train, labels_train_all, train_patients)
    val_ds = MILDataset(fold_bags_val, labels_train_all, val_patients)
    
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    
    # Model
    model = AttentionMIL(input_dim=2048, hidden_dim=256).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    
    best_val_auc = 0
    best_preds = []
    best_pids = []
    epochs_no_improve = 0
    best_model_state = None
    
    for epoch in range(max_epochs):
        train_loss, train_auc = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_auc, preds, pids = evaluate(model, val_loader, criterion)
        
        print(f"Epoch {epoch+1}/{max_epochs} | Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f}")
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_preds = preds
            best_pids = pids
            epochs_no_improve = 0
            import copy
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break
            
    print(f"Fold {fold+1} Best Val AUC: {best_val_auc:.4f}")
    cv_aucs.append(best_val_auc)
    
    # Reload best model for saving
    model.load_state_dict(best_model_state)
    models.append((model, scaler)) # Save model and its associated scaler
    
    # Save OOF
    for pid, pred in zip(best_pids, best_preds):
        oof_preds[pid] = pred
        oof_labels[pid] = labels_train_all[pid]

print("="*40)
print(f"CV Mean AUC: {np.mean(cv_aucs):.4f} +/- {np.std(cv_aucs):.4f}")
global_oof_auc = roc_auc_score([oof_labels[p] for p in oof_preds.keys()], [oof_preds[p] for p in oof_preds.keys()])
print(f"Global OOF AUC: {global_oof_auc:.4f}")
print("="*40)


Building train bags (without averaging tiles)...


Test Bags: 100%|██████████| 149/149 [00:00<00:00, 149.54it/s]



Starting 5-fold CV Training...

--- Fold 1/5 ---
Epoch 1/15 | Train Loss: 0.6665 AUC: 0.5901 | Val Loss: 0.6017 AUC: 0.6970
Epoch 2/15 | Train Loss: 0.5390 AUC: 0.7911 | Val Loss: 0.5644 AUC: 0.7401
Epoch 3/15 | Train Loss: 0.4767 AUC: 0.8410 | Val Loss: 0.5793 AUC: 0.7424
Epoch 4/15 | Train Loss: 0.4163 AUC: 0.8867 | Val Loss: 0.5718 AUC: 0.7448
Epoch 5/15 | Train Loss: 0.3459 AUC: 0.9273 | Val Loss: 0.6231 AUC: 0.7098
Epoch 6/15 | Train Loss: 0.2411 AUC: 0.9734 | Val Loss: 0.7455 AUC: 0.6900
Epoch 7/15 | Train Loss: 0.1693 AUC: 0.9877 | Val Loss: 0.7307 AUC: 0.7191
Epoch 8/15 | Train Loss: 0.1113 AUC: 0.9944 | Val Loss: 0.8357 AUC: 0.7075
Epoch 9/15 | Train Loss: 0.0670 AUC: 0.9976 | Val Loss: 0.8573 AUC: 0.7051
Early stopping triggered at epoch 9
Fold 1 Best Val AUC: 0.7448


TypeError: unhashable type: 'list'

In [ ]:
# ==========================================
# 6) Inference test + submission
# ==========================================
import pandas as pd

print("Running inference on test set using the ensemble of 5 models...")

all_test_preds = []

# Dummy labels for Test DataLoader
dummy_labels = {p: 0 for p in bags_test.keys()}

# Evaluate on all 5 folds and average the predictions
for fold, (model, scaler) in enumerate(models):
    model.eval()
    
    # Scale test bags with the scaler from this fold
    scaled_test_bags = {p: scaler.transform(bags_test[p]) for p in bags_test}
    
    test_ds = MILDataset(scaled_test_bags, dummy_labels, list(bags_test.keys()))
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    
    fold_preds = []
    test_pids = []
    
    with torch.no_grad():
        for bags, labels, pids in test_loader:
            for bag, pid in zip(bags, pids):
                bag = bag.to(device)
                logits, _ = model(bag)
                fold_preds.append(torch.sigmoid(logits).item())
                test_pids.append(pid)
                
    all_test_preds.append(fold_preds)

# Average predictions
avg_test_preds = np.mean(all_test_preds, axis=0)

# Submission
submission = pd.DataFrame({
    "Patient ID": test_pids,
    "Target": avg_test_preds
})

df_test_sample = df_test[["Sample ID", "Patient ID"]].drop_duplicates()
submission_final = df_test_sample.merge(submission, on="Patient ID")[["Sample ID", "Target"]]
submission_final = submission_final.sort_values("Sample ID")

submission_final.to_csv("submission_attention_mil.csv", index=False)
print("Submission saved to 'submission_attention_mil.csv'")
submission_final.head()

Running inference on test set using the ensemble of 5 models...
Submission saved to 'submission_attention_mil.csv'


,Sample ID,Target
0,ID_003.npy,0.410774
1,ID_004.npy,0.573498
2,ID_008.npy,0.180800
3,ID_009.npy,0.263231
4,ID_010.npy,0.192009


In [ ]:
# ==========================================
# 7) Sanity checks
# ==========================================
import matplotlib.pyplot as plt

print("--- Sanity Checks ---")
print(f"Total Unique Patients Train: {len(patients_unique)}")
print(f"Total Unique Patients Test: {len(np.unique(df_test['Patient ID']))}")

tile_counts_train = [len(b) for b in bags_train_all.values()]
print(f"Tiles per bag (Train) - Min: {np.min(tile_counts_train)}, Mean: {np.mean(tile_counts_train):.1f}, Max: {np.max(tile_counts_train)}")

tile_counts_test = [len(b) for b in bags_test.values()]
print(f"Tiles per bag (Test) - Min: {np.min(tile_counts_test)}, Mean: {np.mean(tile_counts_test):.1f}, Max: {np.max(tile_counts_test)}")

# 3 patients attention weights visualization
model = models[0][0]
scaler = models[0][1]
model.eval()

# Select 3 random patients from validation fold 1
sample_pids = val_patients[:3]
scaled_bags = {p: scaler.transform(bags_train_all[p]) for p in sample_pids}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, pid in enumerate(sample_pids):
    bag = torch.tensor(scaled_bags[pid], dtype=torch.float32).to(device)
    with torch.no_grad():
        logits, A = model(bag)
    
    A = A.detach().cpu().flatten()
    pred_prob = torch.sigmoid(logits).item()
    target = labels_train_all[pid]
    
    axes[i].hist(A, bins=50, alpha=0.7)
    axes[i].set_title(f"Patient: {pid}\n Target: {target} | Pred: {pred_prob:.3f}")
    axes[i].set_xlabel("Attention Weight")
    if i == 0:
        axes[i].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


--- Sanity Checks ---


NameError: name 'patients_unique' is not defined